### 행정동과 상권

In [ ]:
import geopandas as gpd
import folium

# 1. 파일 읽기
gdf_market = gpd.read_file('서울시 상권분석서비스(영역-상권).shp')
gdf_admin = gpd.read_file('LSMD_ADM_SECT_UMD_11_202504.shp')

# 2. 좌표계 변환
gdf_market = gdf_market.to_crs(epsg=4326)
gdf_admin = gdf_admin.to_crs(epsg=4326)

# 3. 서울 지도 기본 생성
seoul_map = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

# 4. 상권 경계선 (파란색 + 두껍게)
folium.GeoJson(
    gdf_market,
    name='상권 경계',
    style_function=lambda x: {
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0  # 영역 채우지 않음
    }
).add_to(seoul_map)

# 5. 행정동 경계선 (빨간색 + 얇게)
folium.GeoJson(
    gdf_admin,
    name='행정동 경계',
    style_function=lambda x: {
        'color': 'red',
        'weight': 1.5,
        'fillOpacity': 0
    }
).add_to(seoul_map)

# 6. Layer Control 추가
folium.LayerControl().add_to(seoul_map)

# 7. 지도 출력
seoul_map

### 서울 카페 위치정보

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster

# 1) 데이터 불러오기
gdf_market = gpd.read_file('서울시 상권분석서비스(영역-상권).shp')\
                  .to_crs(epsg=4326)

df = pd.read_csv(
    '소상공인시장진흥공단_상가(상권)정보_서울_202412.csv',
    encoding='utf-8'
)

# 2) “카페”만 필터링 (소분류 기준)
df_cafe = df[df['상권업종소분류명'].str.strip() == '카페'].copy()

# 좌표가 문자열로 돼 있으면 float으로 변환
df_cafe['위도'] = df_cafe['위도'].astype(float)
df_cafe['경도'] = df_cafe['경도'].astype(float)

print(f"커피점 총 {len(df_cafe)}건")  # 제대로 걸렸는지 확인

# 3) Folium 지도 생성
m = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

# (선택) 상권 경계 표시
folium.GeoJson(
    gdf_market,
    name='상권영역',
    style_function=lambda feat: {
        'color': '#444',
        'weight': 1,
        'fillOpacity': 0
    }
).add_to(m)

# 4) MarkerCluster에 카페 표시
marker_cluster = MarkerCluster(name='카페 점포').add_to(m)
for _, row in df_cafe.iterrows():
    folium.Marker(
        [row['위도'], row['경도']],
        popup=f"{row['상호명']} ({row['시군구명']} {row['행정동명']})"
    ).add_to(marker_cluster)

folium.LayerControl(collapsed=False).add_to(m)

m

### 상권의 매출이 높을수록 카페가 많을까?

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup
from folium.plugins import HeatMap, MarkerCluster

# ---------------------- 1. 파일 읽기 ----------------------
# 상권 영역(shapefile)
gdf_market = gpd.read_file('서울시 상권분석서비스(영역-상권).shp') \
                   .to_crs(epsg=4326)

# 행정동 경계(shapefile) – (필요시 팝업용)
gdf_admin = gpd.read_file('LSMD_ADM_SECT_UMD_11_202504.shp') \
                  .to_crs(epsg=4326)

# 카페 원본 데이터
df_cafe_raw = pd.read_csv(
    '소상공인시장진흥공단_상가(상권)정보_서울_202412.csv',
    encoding='utf-8'
)

# 추정 매출 데이터
sales_df = pd.read_csv(
    '서울시 상권분석서비스(추정매출-상권)_2024년.csv',
    encoding='utf-8'
)

# ---------------------- 2. 데이터 전처리 ----------------------
# 2-1) 카페만 필터링 & 좌표 변환
df_cafe = df_cafe_raw.loc[
    df_cafe_raw['상권업종소분류명'].str.strip() == '카페',
    ['상호명','도로명주소','시군구명','행정동명','위도','경도']
].copy()
df_cafe.loc[:, '위도'] = df_cafe['위도'].astype(float)
df_cafe.loc[:, '경도'] = df_cafe['경도'].astype(float)

# 2-2) 상권별 매출 집계
sales_by_market = sales_df.groupby('상권_코드') \
                          .agg({'당월_매출_금액':'sum'}) \
                          .reset_index() \
                          .rename(columns={'당월_매출_금액':'월매출'})
sales_by_market['상권_코드'] = sales_by_market['상권_코드'].astype(str)

# 2-3) shapefile 상권코드 문자열 맞추기
gdf_market['TRDAR_CD'] = gdf_market['TRDAR_CD'].astype(str)

# 2-4) 상권 GeoDataFrame에 매출 머지
gdf_market = gdf_market.merge(
    sales_by_market,
    left_on='TRDAR_CD',
    right_on='상권_코드',
    how='left'
)

# ---------------------- 3. 지도 만들기 ----------------------
m = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

# (1) 상권 경계 (파란색)
folium.GeoJson(
    gdf_market,
    name='상권 경계',
    style_function=lambda feat: {
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0
    }
).add_to(m)

# (2) 상권별 당월 매출 Choropleth
folium.Choropleth(
    geo_data=gdf_market.__geo_interface__,
    data=gdf_market,
    columns=['TRDAR_CD','월매출'],
    key_on='feature.properties.TRDAR_CD',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0,
    legend_name='상권별 추정 월매출(원)',
    nan_fill_color='white'
).add_to(m)

# (3) 카페 점포 위치 – MarkerCluster
marker_cluster = MarkerCluster(name='카페 점포').add_to(m)
for _, row in df_cafe.iterrows():
    folium.Marker(
        [row['위도'], row['경도']],
        popup=f"{row['상호명']} ({row['시군구명']} {row['행정동명']})",
        icon=folium.Icon(color='green', icon='coffee', prefix='fa')
    ).add_to(marker_cluster)

# (4) 카페 밀집도 HeatMap (선택; 기본 숨김)
heat_layer = folium.FeatureGroup(name='카페 HeatMap', show=False)
HeatMap(
    df_cafe[['위도','경도']].values.tolist(),
    radius=10, blur=15, min_opacity=0.3
).add_to(heat_layer)
m.add_child(heat_layer)

# (5) 레이어 컨트롤
folium.LayerControl(collapsed=False).add_to(m)

m

### 카페가 많을수록 매출이 높을까?

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup
from folium.plugins import HeatMap, MarkerCluster

# ---------------------- 1. 파일 읽기 ----------------------
# 상권 영역(shapefile)
gdf_market = gpd.read_file('서울시 상권분석서비스(영역-상권).shp') \
                  .to_crs(epsg=4326)

# 행정동 경계(shapefile) – (팝업용)
gdf_admin = gpd.read_file('LSMD_ADM_SECT_UMD_11_202504.shp') \
                 .to_crs(epsg=4326)

# 카페 원본 데이터
df_cafe_raw = pd.read_csv(
    '소상공인시장진흥공단_상가(상권)정보_서울_202412.csv',
    encoding='utf-8'
)

# 추정 매출 데이터
sales_df = pd.read_csv(
    '서울시 상권분석서비스(추정매출-상권)_2024년.csv',
    encoding='utf-8'
)

# ---------------------- 2. 데이터 전처리 ----------------------
# 2-1) 카페만 필터링 & 좌표 변환
df_cafe = df_cafe_raw.loc[
    df_cafe_raw['상권업종소분류명'].str.strip() == '카페',
    ['상호명','시군구명','행정동명','위도','경도']
].copy()
df_cafe.loc[:, '위도'] = df_cafe['위도'].astype(float)
df_cafe.loc[:, '경도'] = df_cafe['경도'].astype(float)

# 2-2) “커피·음료”만 필터링해서 상권별 매출 집계
bev_df = sales_df.loc[
    sales_df['서비스_업종_코드_명'].str.contains('커피|음료', na=False),
    ['상권_코드','당월_매출_금액']
].copy()

sales_by_market = bev_df.groupby('상권_코드')['당월_매출_금액'] \
                        .sum() \
                        .reset_index() \
                        .rename(columns={'당월_매출_금액':'커피음료_매출'})
sales_by_market['상권_코드'] = sales_by_market['상권_코드'].astype(str)

# 2-3) shapefile 상권코드 문자열 맞추기
gdf_market['TRDAR_CD'] = gdf_market['TRDAR_CD'].astype(str)

# 2-4) 상권 GeoDataFrame에 매출 머지
gdf_market = gdf_market.merge(
    sales_by_market,
    left_on='TRDAR_CD',
    right_on='상권_코드',
    how='left'
)

# ---------------------- 3. 지도 만들기 ----------------------
m = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

# (1) 상권 경계 (파란색)
folium.GeoJson(
    gdf_market,
    name='상권 경계',
    style_function=lambda feat: {
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0
    }
).add_to(m)

# (2) 상권별 커피·음료 월매출 Choropleth
folium.Choropleth(
    geo_data=gdf_market.__geo_interface__,
    data=gdf_market,
    columns=['TRDAR_CD','커피음료_매출'],
    key_on='feature.properties.TRDAR_CD',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0,
    legend_name='커피·음료 추정 월매출 (원)',
    nan_fill_color='white'
).add_to(m)

# (3) 카페 점포 위치 – MarkerCluster
marker_cluster = MarkerCluster(name='카페 점포').add_to(m)
for _, row in df_cafe.iterrows():
    folium.Marker(
        [row['위도'], row['경도']],
        popup=f"{row['상호명']} ({row['시군구명']} {row['행정동명']})",
        icon=folium.Icon(color='green', icon='coffee', prefix='fa')
    ).add_to(marker_cluster)

# (4) 카페 밀집도 HeatMap (선택; 기본 숨김)
heat_layer = folium.FeatureGroup(name='카페 HeatMap', show=False)
HeatMap(
    df_cafe[['위도','경도']].values.tolist(),
    radius=10, blur=15, min_opacity=0.3
).add_to(heat_layer)
m.add_child(heat_layer)

# (5) 행정동별 팝업용 레이어
folium.GeoJson(
    gdf_admin,
    name='행정동 경계',
    style_function=lambda feat: {
        'color': 'red',
        'weight': 1,
        'fillOpacity': 0
    },
    popup=GeoJsonPopup(
        fields=['EMD_NM'],
        aliases=['행정동명'],
        localize=True
    )
).add_to(m)

# (6) 레이어 컨트롤
folium.LayerControl(collapsed=False).add_to(m)

m